# LLM Cycle-Geometry — Task 15 Full-Model Run (Colab)

Runs Part 1 (circle reproduction, the validation gate) and Part 2 (the AUC comparison ladder) against a selectable model (`gemma-2-2b`, `gemma-3-12b`, or `llama-3.1-8b` — see §2b).

**Before running:** Runtime → Change runtime type → pick a GPU. `gemma-2-2b` fits a free **T4**; the 8B/12B models need a Colab Pro **L4/A100**.

## 1. Mount Google Drive and install the package wheel

The wheel (`networkgeometry-0.1.14-py3-none-any.whl`) lives in your Drive folder `NetworkGeometry-Colab/`. This mounts your Drive (one browser authorization click) and installs directly from there — no manual upload needed.

**0.1.14 (bug fix):** large models (`gemma-3-12b`, `llama-3.1-8b`) now load in **bfloat16**, not fp16. Gemma/Llama have massive outlier activations that overflow fp16 (→ `inf` → `SVD did not converge` in Part 1); bf16 has fp32's exponent range and is these checkpoints' native dtype. `gemma-2-2b` keeps fp16 (its activations fit). Extraction now also raises a clear, actionable message if any activation is non-finite instead of failing later inside the SVD.

**0.1.13:** the model is now **selectable** (§2b) — `gemma-2-2b`, `gemma-3-12b` (Gemma 3 12B), or `llama-3.1-8b`, all base/pretrained via TransformerLens; `llama-4-scout` is listed but raises a clear "unsupported" error. The layer list and a per-model `results/<model>/` folder are derived automatically, and the **model name is written into every results file (`"model"` field) and every figure title**.

**0.1.12:** added **pool 2** — a comprehension-probe pool (§7b). Category-declaring, per-structure frames (`"The day of the week is {X}"`, `"The animal is {X}"`, …) that end in the state word but imply no ordering; a cell generates the model's continuation for every state so you can read the responses. The **hierarchy** control is now 3 animal families (mammals/birds/fish, 9 leaves).

**0.1.11:** strict / paper-exact leg also plots a **state correlation matrix** per cyclic structure at layers 6, 12, 14, 15.

**0.1.10:** the Part 2 AUC-ladder figure's **caption** spells out the concrete prompts each comparison uses.

**0.1.9:** shared/neutral pool rewritten to natural, topic-introducing frames varied by sentence type.

**0.1.2 fixed a real bug:** templates ended in a period after `{X}`, so extraction read the period's activation. Templates are now state-word-final. **Any results from wheels ≤0.1.1 should be discarded.**

**When you rebuild the wheel after a code fix, the version bumps (…0.1.13 → 0.1.14 …) so the filename always changes.** Delete the old versioned wheel from Drive and drop in the new one — never overwrite a same-named file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/NetworkGeometry-Colab'
WHEEL_PATH = f'{DRIVE_DIR}/networkgeometry-0.1.14-py3-none-any.whl'
print('Wheel path:', WHEEL_PATH)

In [ ]:
import numpy, scipy
NUMPY_VER = numpy.__version__
SCIPY_VER = scipy.__version__
print(f'Pinning Colab stock numpy=={NUMPY_VER} scipy=={SCIPY_VER} so nothing upgrades them')

!pip install -q "{WHEEL_PATH}" "numpy=={NUMPY_VER}" "scipy=={SCIPY_VER}" transformer_lens
!pip uninstall -y -q torchvision
!pip uninstall -y -q torchaudio

import importlib.metadata
installed_ver = importlib.metadata.version('networkgeometry')
expected_ver = WHEEL_PATH.split('networkgeometry-')[1].split('-py3')[0]
assert installed_ver == expected_ver, (
    f'Installed networkgeometry=={installed_ver} but expected =={expected_ver} — '
    'the wheel on Drive is stale, re-upload it.'
)
print(f'Confirmed networkgeometry=={installed_ver} installed')

**Root cause of the numpy/scipy crash, actually found:** `pip install transformer_lens` doesn't go through this project's lockfile at all — it's a fresh PyPI resolution every time, independent of the relaxed floors in `pyproject.toml` (those only govern our own wheel's declared requirements, which are already satisfied and never force an upgrade). If `transformer_lens`/`transformers` on PyPI have since bumped their own numpy/scipy floors, installing them can silently upgrade Colab's preinstalled numpy to a version `sklearn` (also preinstalled, imported transitively by `transformers.generation.candidate_generator`) wasn't built against, causing `AttributeError: ... has no attribute '_blas_supports_fpe'`. This is why the fix kept "working" on one VM and breaking again on the next.

The cell above now captures Colab's stock numpy/scipy versions before installing anything, then pins those exact versions in the same `pip install` call as `transformer_lens` — so pip's resolver can't upgrade them no matter what transformer_lens's own metadata requests, keeping Colab's internally-consistent preinstalled numpy/scipy/sklearn trio intact.

Two follow-on version-skew issues also showed up, both fixed by the cell above:

- `torchvision::nms` op-registration failure — Colab's preinstalled `torchvision` doesn't match whatever `torch` gets installed alongside `transformer_lens`. We don't need vision utilities for a text-only Gemma model, so the cell above uninstalls `torchvision` outright.
- `RuntimeError: PyTorch and TorchAudio were compiled with different CUDA versions` — same story for `torchaudio`, which we also don't need. Uninstalled too.

**Important:** any time you change runtime type (GPU, RAM tier, etc.), Colab gives you a brand-new VM with none of this applied — you must re-run the cell above from scratch on that new VM. A plain Runtime to Restart session (no runtime-type change) keeps the same VM/disk, so pip state survives; you'd only need to restart if you still have transformers/torch imported in memory from a prior attempt in this session (stale in-memory "is this package available" caches won't reflect an uninstall until the process restarts).

## 2b. Choose the model

Pick which model to run. The pipeline loads it through TransformerLens `HookedTransformer` (that's how it reads `blocks.{l}.hook_resid_post`), so only TransformerLens-supported **base** models are offered:

| key | model | layers | dtype | GPU |
|---|---|---|---|---|
| `gemma-2-2b` *(default)* | `google/gemma-2-2b` | 26 | fp16 | free T4 (~4 GB) |
| `gemma-3-12b` | `google/gemma-3-12b-pt` | 48 | bf16 | A100 (~24 GB) — Colab Pro |
| `llama-3.1-8b` | `meta-llama/Llama-3.1-8B` | 32 | bf16 | L4 / A100 (~16 GB) — Colab Pro |
| `llama-4-scout` | — | — | — | **unsupported** by TransformerLens 3.5.1 — selecting it raises a clear error |

The 12B "Gemma" is officially **Gemma 3 12B** (there is no Gemma 4); it is labelled `gemma-3-12b` in every results file and figure title. Large models load in **bfloat16** (fp16 overflows their massive activations → `SVD did not converge`); `gemma-2-2b` keeps fp16. The layer list and `results/<model>/` folder are derived automatically.

**Licenses:** accept each model's license on Hugging Face under your account before loading.

In [ ]:
from networkgeometry.models import MODELS, resolve_model

# ---- pick your model here ----
MODEL_KEY = 'gemma-2-2b'   # 'gemma-2-2b' | 'gemma-3-12b' | 'llama-3.1-8b'   (llama-4-scout: unsupported)
# ------------------------------

MODEL = resolve_model(MODEL_KEY)   # raises a clear error for an unknown/unsupported key
print(f'Selected: {MODEL.key}  ->  {MODEL.hf_id}  ({MODEL.n_layers} layers, {MODEL.dtype}, fits {MODEL.fits})\n')
print('Menu:')
for key, choice in MODELS.items():
    status = choice.fits if choice.supported else 'UNSUPPORTED by TransformerLens 3.5.1'
    print(f'  {key:14s} {(choice.hf_id or "-"):26s} {status}')

## 2. Authenticate with Hugging Face

Runs an interactive login widget — paste your token (the same one used locally, or a fresh one). You must have already accepted the license at https://huggingface.co/google/gemma-2-2b under this account.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import model_info
info = model_info(MODEL.hf_id)
print('Access confirmed:', info.id, '| gated:', info.gated)

## 3. Load the selected model (GPU, no-processing path)

Loads via `networkgeometry.models.load_model`, which uses `from_pretrained_no_processing` (skips the memory-heavy weight processing that OOM'd the default path) at each model's chosen dtype: **bfloat16** for `gemma-3-12b` / `llama-3.1-8b` (their native precision — fp16 overflows their massive outlier activations and Part 1 then dies with `SVD did not converge`), **float16** for `gemma-2-2b` (validated, fits fp16). If this OOMs, switch to a bigger runtime (Runtime → Change runtime type).

In [ ]:
import torch, time
print('CUDA available:', torch.cuda.is_available())

from networkgeometry.models import load_model
t0 = time.perf_counter()
model = load_model(MODEL_KEY, device='cuda' if torch.cuda.is_available() else 'cpu')
print(f'{MODEL_KEY} ({MODEL.hf_id}) loaded in {time.perf_counter()-t0:.1f}s '
      f'as {MODEL.dtype}, {model.cfg.n_layers} layers, d_model={model.cfg.d_model}')

## 4. Run Part 1 — circle reproduction (the validation gate)

Per spec §4.5: if month/day circularity doesn't reproduce cleanly here, treat it as a pipeline bug and debug before trusting Part 2 — do not proceed on a weak Part 1 result.

In [ ]:
from networkgeometry.run import run_part1
import time

LAYERS = list(range(model.cfg.n_layers))   # all layers of the selected model (26 / 32 / 48)
OUT_DIR = f'results/{MODEL_KEY}'            # per-model folder so runs don't overwrite each other

t0 = time.perf_counter()
part1_results = run_part1(model, LAYERS, OUT_DIR)
print(f'Part 1 done in {time.perf_counter()-t0:.1f}s')

for name, layer_results in part1_results.items():
    print(f'\n=== {name} ===')
    for lc in layer_results:
        print(f'  layer {lc.layer:2d}: residual={lc.normalized_residual:.4f}  '
              f'angular_order={lc.angular_order:.4f}  top2_var_ratio={lc.top2_variance_ratio:.4f}')

**Stop and inspect before continuing.** Look for layers where `angular_order` and `top2_variance_ratio` are both high (>0.9-ish) for both day and month — those are the clean, reproduced-circle layers. `results/part1_summary.json` now has these numbers on disk too.

## 5. Plot Part 1 manifolds (optional, for visual inspection)

In [ ]:
from networkgeometry.figures.part1_plots import plot_circularity_by_layer
import matplotlib.pyplot as plt
from IPython.display import Image, display

for name, layer_results in part1_results.items():
    path = plot_circularity_by_layer(
        layer_results, f'{OUT_DIR}/{name}_circularity_by_layer.png',
        title=f'{name} circularity vs layer  ·  model: {MODEL_KEY}')
    display(Image(filename=path))

## 5b. Run Part 1 — strict leg (paper's exact templates)

Per spec §4.3(a): reproduces the paper's own figures literally — one prompt per structure, no template averaging: `"The month of the year is {X}"`, `"The day of the week is {X}"` (day has no paper equivalent; we use the same construction pattern). Years isn't included here yet — it needs a different, not-yet-implemented uncentered/Toeplitz geometry analysis rather than the circular fit used for day/month.

In [ ]:
from networkgeometry.run import run_part1_strict

t0 = time.perf_counter()
part1_strict_results = run_part1_strict(model, LAYERS, OUT_DIR)
print(f'Part 1 (strict leg) done in {time.perf_counter()-t0:.1f}s')

for name, layer_results in part1_strict_results.items():
    print(f'\n=== {name} (strict) ===')
    for lc in layer_results:
        print(f'  layer {lc.layer:2d}: residual={lc.normalized_residual:.4f}  '
              f'angular_order={lc.angular_order:.4f}  top2_var_ratio={lc.top2_variance_ratio:.4f}')

### 5b(ii). Strict-leg plots — the paper's exact-context replication

Three figures per cyclic structure, computed from the paper-exact single template (`"The month of the year is {X}"`, `"The day of the week is {X}"`):

- **PC1–PC2 ring manifold** — the paper's iconic figure: each state projected onto the top 2 principal components; a clean, calendar-ordered ring is the reproduction target. Shown at layers 6, 12, 15. (Polysemous months May/March/August are dropped from the basis, per spec §4.3b.)
- **State correlation matrix** — the Pearson correlation between states' residual-stream vectors for this same strict template (all states included), shown at layers 6, 12, 14, 15. This is the strict-leg counterpart of §5c's shared-pool heatmaps: expect the same **circulant banding** for day/month, now from the paper's exact context rather than the averaged pool.
- **Circularity vs layer** — angular order and top-2 variance ratio across all 26 layers for the strict template, so you can see which layer's ring is cleanest.

In [ ]:
from networkgeometry.run import mean_state_matrices
from networkgeometry.figures.part1_plots import (
    plot_circularity_by_layer, plot_manifold, plot_correlation_matrix,
    STRICT_CORRELATION_CAPTION)
from networkgeometry.geometry.part1 import manifold_scores
from networkgeometry.stimuli.definitions import load_structures

RING_LAYERS = [6, 12, 15]
STRICT_CORR_LAYERS = [6, 12, 14, 15]
structures = load_structures()
strict_mats = mean_state_matrices(
    model, ['day', 'month'], sorted(set(RING_LAYERS + STRICT_CORR_LAYERS)), pool='strict')

for name in ['day', 'month']:
    labels = strict_mats[name]['labels']
    excluded = structures[name].excluded
    for layer in RING_LAYERS:
        scores, kept = manifold_scores(strict_mats[name]['matrices'][layer], labels, excluded=excluded)
        path = plot_manifold(
            scores, kept, f'{OUT_DIR}/{name}_strict_ring_layer{layer}.png',
            title=f'{name} PC1-PC2 ring, layer {layer} (strict)  ·  model: {MODEL_KEY}')
        display(Image(filename=path))
    for layer in STRICT_CORR_LAYERS:
        path = plot_correlation_matrix(
            strict_mats[name]['matrices'][layer], labels,
            f'{OUT_DIR}/{name}_strict_corr_layer{layer}.png',
            title=f'{name} — state correlation, layer {layer} (strict)  ·  model: {MODEL_KEY}',
            caption=STRICT_CORRELATION_CAPTION)
        display(Image(filename=path))
    path = plot_circularity_by_layer(
        part1_strict_results[name], f'{OUT_DIR}/{name}_strict_circularity_by_layer.png',
        title=f'{name} circularity vs layer (strict)  ·  model: {MODEL_KEY}')
    display(Image(filename=path))

## 5c. State correlation matrices at layers 6, 12, 14, 15

For each of the five structures and each layer, the Pearson correlation matrix between states' residual-stream activation vectors (averaged over the shared-pool templates, all states included). The layer is written into every figure's title, and the caption defines the quantity. For the cyclic structures (day, month) expect **circulant banding** — each state most correlated with its cyclic neighbours; for `hierarchy` expect **nested blocks**, and for `flat` expect no structure beyond the unit diagonal.

In [ ]:
from networkgeometry.run import mean_state_matrices
from networkgeometry.figures.part1_plots import plot_correlation_matrix

CORR_LAYERS = [6, 12, 14, 15]
STRUCTURES = ['day', 'month', 'years', 'hierarchy', 'flat']

corr_mats = mean_state_matrices(model, STRUCTURES, CORR_LAYERS)
for layer in CORR_LAYERS:
    for name in STRUCTURES:
        labels = corr_mats[name]['labels']
        matrix = corr_mats[name]['matrices'][layer]
        path = plot_correlation_matrix(
            matrix, labels, f'{OUT_DIR}/{name}_corr_layer{layer}.png',
            title=f'{name} — state correlation, layer {layer}  ·  model: {MODEL_KEY}')
        display(Image(filename=path))

## 6. Run Part 2 — the AUC comparison ladder

Set `LAYERS` below to the gate-passing layers you identified in Part 1 (or leave as the same full range — `run_part2` applies its own within-structure gate internally per layer, so passing all 26 is safe, just slower).

Per spec §5.3, this produces one row per (comparison, layer) covering the full Stage-2 table: `day (within)` / `month (within)` (the gate reference), `day -> month` / `month -> day` in both **matched** and **specific** forms, and the `day -> years` / `day -> hierarchy` / `day -> flat` controls. Cross rows only appear at layers where their source structure passed its own within-structure significance gate.

**matched vs specific** (this is the key distinction the two cross-cycle rows test):
- **matched** = the *shared/neutral* template pool — the **same** frame is used for both structures, so day and month appear in identical wording (e.g. `"Tell me about {X}"`). These frames are deliberately preposition-free so one frame works identically for days, months, years and nouns. This asks: do day and month share a subspace *when the surrounding context is held constant*?
- **specific** = each structure's *own* natural frame, which necessarily differs in wording (day `"We'll meet on {X}"` vs month `"We'll meet in {X}"`). This asks: does the sharing *survive a change of context* — i.e. is it abstract, not phrasing-bound?

In [ ]:
from networkgeometry.run import run_part2

t0 = time.perf_counter()
part2_results = run_part2(model, LAYERS, OUT_DIR)
print(f'Part 2 done in {time.perf_counter()-t0:.1f}s')

for row in sorted(part2_results, key=lambda r: (r.label, r.layer)):
    p_str = (f'raw_p={row.raw_p:.4f} fdr_p={row.fdr_p:.4f} bonferroni_p={row.bonferroni_p:.4f}'
             if row.raw_p is not None else '(within reference, not corrected)')
    print(f'{row.label:28s} layer {row.layer:2d}: auc={row.auc:.4f} sem={row.sem:.4f}  {p_str}')

`results/summary.json` now has the full ladder on disk.

## 7. Plot Part 2 results

One AUC-vs-layer curve per Stage-2 comparison row (spec §5.3 table), chance line at 0.5.

In [ ]:
from networkgeometry.figures.part2_plots import plot_stage2_ladder
from networkgeometry.stimuli.definitions import load_templates

templates = load_templates()
path = plot_stage2_ladder(
    part2_results, f'{OUT_DIR}/stage2_ladder.png', templates=templates, model_name=MODEL_KEY)
display(Image(filename=path))

## 7b. Comprehension-probe pool (pool 2) — read the model's responses

Pool 2 (spec §3.4·3) uses **category-declaring** frames that end in the state word but say nothing about ordering (no "after / before / next"), so they can't manufacture the ring geometry they're used to probe. The frames name the *kind* — `"The day of the week is {X}"`, `"The animal is {X}"` — and we let the model continue, so you can **read the responses** and judge for yourself whether it recognized each concept (Monday as a weekday, salmon as a fish). Greedy decoding, 12 new tokens per prompt.

The hierarchy control is now three animal families (mammals: dog/cat/horse, birds: crow/eagle/owl, fish: salmon/trout/tuna) so `"The animal is {X}"` fits every leaf.

In [ ]:
from networkgeometry.stimuli.definitions import load_structures, load_templates

_structures = load_structures()
_probe = load_templates()['probe']

def show_probe_responses(max_new_tokens=12):
    for name in ['day', 'month', 'years', 'hierarchy', 'flat']:
        states = sorted(_structures[name].states, key=lambda s: s.canonical_index)
        print(f"\n{'='*72}\n{name.upper()}\n{'='*72}")
        for frame in _probe[name]:
            print(f"\n--- frame: {frame} ---")
            for state in states:
                prompt = frame.replace('{X}', state.label)
                text = model.generate(prompt, max_new_tokens=max_new_tokens,
                                      do_sample=False, verbose=False)
                print(text)

show_probe_responses()

### 7b(ii). Part 2 on the probe pool

The same Stage-2 ladder (§5.3) computed in the **probe context**. Because the probe frames are per-structure (there is no shared frame), each cross-cycle test appears as a single row — `day -> month`, `month -> day` — with **no matched/specific split**; the within gate (`day (within)`, `month (within)`) and the `day -> years / hierarchy / flat` controls are all evaluated in the probe context. Reported alongside the pool-1 ladder above, it asks the narrower question: *given prompts where we can read off that the model recognizes each state, is the cycle code still shared?*

In [ ]:
from networkgeometry.run import run_part2_probe
from networkgeometry.figures.part2_plots import plot_stage2_ladder
import time

t0 = time.perf_counter()
probe_results = run_part2_probe(model, LAYERS, OUT_DIR)
print(f'Part 2 (probe pool) done in {time.perf_counter()-t0:.1f}s')

for row in sorted(probe_results, key=lambda r: (r.label, r.layer)):
    p_str = (f'raw_p={row.raw_p:.4f} fdr_p={row.fdr_p:.4f} bonferroni_p={row.bonferroni_p:.4f}'
             if row.raw_p is not None else '(within reference, not corrected)')
    print(f'{row.label:20s} layer {row.layer:2d}: auc={row.auc:.4f} sem={row.sem:.4f}  {p_str}')

probe_note = (
    "Pool 2 = comprehension-probe frames (category-declaring, no ordering cue), per-structure, "
    "so each cross-cycle test is a single row (no matched/specific split). Frames: "
    "day 'The day of the week is {X}', month 'The month of the year is {X}', "
    "years 'The year is {X}', hierarchy 'The animal is {X}', flat 'This is a {X}'."
)
path = plot_stage2_ladder(
    probe_results, f'{OUT_DIR}/stage2_ladder_probe.png', context_note=probe_note, model_name=MODEL_KEY)
display(Image(filename=path))

## 8. Download results back to your machine

In [ ]:
import shutil
from google.colab import files

archive = f'task15_results_{MODEL_KEY}'
shutil.make_archive(archive, 'zip', OUT_DIR)
files.download(f'{archive}.zip')

# Also persist a copy to the same Drive folder
shutil.copy(f'{archive}.zip', f'{DRIVE_DIR}/{archive}.zip')
print(f'Results also saved to Drive: {DRIVE_DIR}/{archive}.zip')